# **Scattering correction for infrared spectra of biological cells using computational infrared microspectroscopy and deep learning**

*Sergio G. Rodrigo $^{1,2*}$, Ilia L. Rasskazov $^{3}$, Luis Martin-Moreno $^{1,4}$, Martin Schnell $^{5,6,7*}$*



+ 1 Instituto de Nanociencia y Materiales de Aragón (INMA), CSIC-Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 2 Departamento de Física Aplicada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 3 Independent Researcher, San Jose, California 95124, United States
+ 4 Departamento de Física de la Materia Condensada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 5 CIC nanoGUNE BRTA, 20018 Donostia-San Sebastián, Spain
+ 6 Donostia International Physics Center, Donostia-San Sebastian 20018, Spain
+ 7 IKERBASQUE, Basque Foundation for Science, 48013 Bilbao, Spain

$^*$ Corresponding authors. Email: sergut@unizar.es (SGR), schnelloptics@gmail.com (MS)

# *Description*

Presented in Fig. 3(a), the fully connected encoder-decoder neural network implemented in this notebook predict true absorption (heat dissipation) spectra, $\mathcal{A}_{true} (\nu)$, from absorbance spectra, $A(\nu)$ . Both the input and output dimensions corresponded to the number of spectral wavelengths (N_wav=251). The network architecture consisted of an encoder–decoder structure with three encoding layers and three decoding layers.

# Main Packages

In [ ]:
# Common packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load matlab files
import scipy.io

# Neural Networks packages
import tensorflow as tf
from tensorflow.keras import models,layers,Model,callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, Adam

# Auxiliary Plotting Functions

In [ ]:
def plot_history_by_key(history):  
  import matplotlib.pyplot as plt
  keys=history.history.keys()
  print(keys)
  for key in keys:
    plt.loglog(history.history[key],marker='o',markersize=0.0, linewidth=1.0,label=key)
  plt.xlabel('epoch')
  plt.legend()
  plt.show()
  pass

def plot_nn_vs_true(wlength_range,nn_results,true_results,output_name):
  index=wlength_range
  fig, ax = plt.subplots(2,1,dpi=100, figsize=(8,8)) #sharex='col', sharey='row',
  ax[0].plot(index,nn_results, color="blue", label="Neural Network",fillstyle='none',linewidth=0, marker='.', markersize=15)
  ax[0].plot(index,true_results, color="red", label="True values",linewidth=0, marker='.', markersize=10)  
  txt1 ="Wavelength $(cm)^{-1}$"
  txt2 = output_name
  ax[0].set_xlabel(txt1)
  ax[0].set_ylabel(txt2)
  ax[0].legend()

  ax[1].plot(true_results,true_results, color="blue", fillstyle='none',linewidth=2, marker='.', markersize=0)
  ax[1].plot(nn_results,true_results, color="red", linewidth=0, marker='.', markersize=10)
  ax[1].set_aspect(1.0/ax[1].get_data_ratio(), adjustable='box')
  txt1 = output_name+' - Neural Network'
  txt2 = output_name+' - True values'
  ax[1].set_xlabel(txt1)
  ax[1].set_ylabel(txt2)  
  txt="Neural Network vs True values" # Final wavelength)
  fig.suptitle(txt)
  fig.show

# Load the Data

In [ ]:
# Load the Training and Validation Datasets
path="./data/training_data/"
collect = scipy.io.loadmat(path+'fdtd_simulations.mat')

# Wavelengths
wlength=collect['vec_wn']

# Spectra
absorbance=collect['collect_spec_norm'].T
absorption=collect['collect_frac_abs'].T

# Split Into Training and Validation Sets

In [ ]:
# CALL ONCE!!!
shuffle_list = np.random.permutation(absorbance.shape[0]) 

## INPUT: Absorbance = $A(\nu)$ 

In [ ]:
# Absorbance
spec= absorbance[shuffle_list]
nsamples=spec.shape[0]
ntrain0 = int(nsamples*0.9)
nvalidation0 = nsamples-ntrain0
ndata = ntrain0 + nvalidation0

# Training and validation datasets
x_train_absorbance = spec[0:ntrain0,:]
x_val_absorbance = spec[ntrain0:ntrain0 + nvalidation0, :]

## OUTPUT: True Absorption = $\mathcal{A}^{ED}(\nu)$  

In [ ]:
# Absorption
spec= absorption[shuffle_list]
nsamples=spec.shape[0]
ntrain0 = int(nsamples*0.9)
nvalidation0 = nsamples-ntrain0
ndata = ntrain0 + nvalidation0

# Training and validation datasets
y_train_absorption = spec[0:ntrain0,:]
y_val_absorption = spec[ntrain0:ntrain0 + nvalidation0, :]

# Train the Neural Network

In [ ]:
def build_autoencoder(y,nneurons,nn2,nnmin,verbose=1):
  # kernel and bias initialization
  kernel_init = tf.keras.initializers.GlorotNormal()
  bias_init =   tf.keras.initializers.GlorotUniform()

  # Encoder
  input_encoder = Input(shape=(y.shape[1],))
  encoder_ly1 = Dense(nneurons, activation='relu')(input_encoder)                      
  x = Dense(nn2, activation='relu')(encoder_ly1)
  encoder = Dense(nnmin, 
                       activation='relu')(x)
  
  # Decoder  
  decoder_ly1 = Dense(nn2, activation='relu')(encoder)
  x = Dense(nneurons, activation='relu')(decoder_ly1)
  decoder = Dense(y.shape[1], activation=None,\
                            kernel_initializer = kernel_init,\
                            bias_initializer = bias_init)(x)
  

  # Autoencoder  
  model=Model(input_encoder,decoder)  
  
  if(verbose==1):  
    model.summary()
  return model,encoder,decoder

def call_backs(model_name):
    callbacks_ = [tf.keras.callbacks.EarlyStopping(monitor='loss',
                                            patience=1000,
                                            restore_best_weights=True),
                 callbacks.ModelCheckpoint(model_name,save_best_only=True)]

    model.summary()
    
    return callbacks_

In [ ]:
# Model name
model_name='encoder_decoder_network_model.keras'
print(model_name)

In [ ]:
# Epochs
epochs = 1000
# Hyperparameters
batch_size = 32
# Define the loss and optimizer
loss_function = 'mse' #tf.keras.losses.CosineSimilarity () #'mse'
opt = Adam()

nneurons = 50
nn2 = 25
nnmin = 4 #10# Neurons latent space

model,encoder,decoder=build_autoencoder(x_train_absorbance,nneurons,nn2,nnmin)

# Compile and fit
model.compile(optimizer=opt, loss=loss_function)

history = model.fit(x_train_absorbance, 
                    y_train_absorption,
                    epochs=epochs,
                    batch_size = batch_size,                    
                    validation_data=(x_val_absorbance, 
                                     y_val_absorption),
                    callbacks=call_backs(model_name))

plot_history_by_key(history)

# Neural Network Predictions vs. Ground Truth

## True vs Ground

In [ ]:
# Load model
model_name='encoder_decoder_network_model.keras'
model=models.load_model(model_name)

# Predictions vs true values
nn_results=model.predict(x_val_absorbance,verbose=0)
true_results=y_val_absorption

plt.scatter(true_results,nn_results,alpha=0.1,color='r')
plt.plot(y_val_absorption,y_val_absorption,color='blue')
plt.xlabel("True values")
plt.ylabel("Predictions")
plt.show()

## Compare spectra

In [ ]:
# Load model
model_name='encoder_decoder_network_model.keras'
model=models.load_model(model_name)

# Pick a random spectrum from the validation dataset  
j= np.random.randint(x_val_absorbance.shape[0]) 
print('index:',j)
results=model.predict(x_val_absorbance,verbose=0)
nn_results=results[j,:]
true_results=y_val_absorption[j,:]

# Plot predictions vs true values
plot_nn_vs_true(wlength,nn_results,true_results,"True absorption")